In [20]:
import pandas as pd
import json

# load JSON instead of CSV
with open("dataset_indeed-scraper.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(2))

Rows: 100
Columns: ['title', 'companyName', 'location', 'age', 'datePublished', 'jobType', 'companyUrl', 'rating', 'descriptionText', 'descriptionHtml', 'benefits', 'occupation', 'attributes', 'workingSystem', 'source', 'hiringDemand.isUrgentHire', 'hiringDemand.isHighVolumeHiring', 'locale', 'salary', 'expired', 'postedToday', 'scrapingInfo.page', 'scrapingInfo.index', 'jobKey', 'jobUrl', 'emails', 'isRemote', 'socialInsurance', 'shiftAndSchedule', 'applyUrl', 'requirements', 'companyLogoUrl', 'companyHeaderUrl', 'numOfCandidates']
                                               title   companyName  \
0                      Analyste d'exploitation - F/H       INZERTY   
1  Stage 2 mois – Data Analyst / Lead Generation ...  Vroom Market   

                                            location          age  \
0  {'city': 'Lille', 'postalCode': '59000', 'coun...  10 days ago   
1  {'city': 'Télétravail', 'postalCode': None, 'c...  a month ago   

  datePublished               jobType     

In [25]:
import pandas as pd
import sqlite3
import json

# ── 1. LOAD ──────────────────────────────────────────────────────────────
with open("dataset_indeed-scraper.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Raw rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}\n")

# ── 2. FLATTEN NESTED FIELDS ─────────────────────────────────────────────
# location dict → city, country
df["city"] = df["location"].apply(
    lambda x: x.get("city") if isinstance(x, dict) else None
)
df["country"] = df["location"].apply(
    lambda x: x.get("country") if isinstance(x, dict) else None
)

# rating dict → number
df["company_rating"] = df["rating"].apply(
    lambda x: x.get("rating") if isinstance(x, dict) else None
)

# jobType list → first value as string
df["job_type"] = df["jobType"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else "Non spécifié"
)

# ── 3. DATES ──────────────────────────────────────────────────────────────
df["date_published"] = pd.to_datetime(df["datePublished"], errors="coerce")
df["month_posted"]   = df["date_published"].dt.to_period("M").astype(str)
df["day_of_week"]    = df["date_published"].dt.day_name()

# ── 4. SALARY FLAG ────────────────────────────────────────────────────────
df["has_salary"] = df["salary"].apply(
    lambda x: bool(x) and str(x) not in ["[]", "", "None"]
)

# ── 5. SELECT & RENAME COLUMNS ───────────────────────────────────────────
keep = [
    "title", "companyName", "city", "country",
    "job_type", "date_published", "month_posted", "day_of_week",
    "isRemote", "salary", "has_salary", "company_rating",
    "occupation", "jobUrl", "source"
]
df = df[[c for c in keep if c in df.columns]]
df = df.rename(columns={"companyName": "company", "isRemote": "is_remote"})

# ── 6. DEDUPLICATE & DROP NULLS ───────────────────────────────────────────
df = df.dropna(subset=["title", "company"])
df = df.drop_duplicates(subset=["title", "company", "city"])
df["date_scraped"] = pd.Timestamp.today().strftime("%Y-%m-%d")

print(f"Clean rows: {len(df)}")
print(df.head(3))

for col in df.columns:
    df[col] = df[col].apply(
        lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
    )

# now save to SQLite
conn = sqlite3.connect("jobs.db")
df.to_sql("job_listings", conn, if_exists="replace", index=False)
print("Saved to SQLite successfully")

# ── 7. STORE IN SQLITE ────────────────────────────────────────────────────
conn = sqlite3.connect("jobs.db")
df.to_sql("job_listings", conn, if_exists="replace", index=False)

print("\n--- SQL VERIFICATION ---")

print("\nTotal jobs:")
print(pd.read_sql("SELECT COUNT(*) as total FROM job_listings", conn))

print("\nTop 10 companies:")
print(pd.read_sql("""
    SELECT company, COUNT(*) as job_count
    FROM job_listings
    GROUP BY company
    ORDER BY job_count DESC
    LIMIT 10
""", conn))

print("\nTop 10 cities:")
print(pd.read_sql("""
    SELECT city, COUNT(*) as job_count
    FROM job_listings
    WHERE city IS NOT NULL
    GROUP BY city
    ORDER BY job_count DESC
    LIMIT 10
""", conn))

print("\nJobs by type:")
print(pd.read_sql("""
    SELECT job_type, COUNT(*) as count
    FROM job_listings
    GROUP BY job_type
    ORDER BY count DESC
""", conn))

print("\nRemote vs onsite:")
print(pd.read_sql("""
    SELECT 
        CASE WHEN is_remote = 1 THEN 'Remote' ELSE 'Onsite' END as work_type,
        COUNT(*) as count
    FROM job_listings
    GROUP BY is_remote
""", conn))

conn.close()

# ── 8. EXPORT CSV FOR POWER BI ────────────────────────────────────────────
df.to_csv("jobs_clean.csv", index=False)
print("\nDone — jobs.db and jobs_clean.csv ready for Power BI")

Raw rows: 100
Columns: ['title', 'companyName', 'location', 'age', 'datePublished', 'jobType', 'companyUrl', 'rating', 'descriptionText', 'descriptionHtml', 'benefits', 'occupation', 'attributes', 'workingSystem', 'source', 'hiringDemand.isUrgentHire', 'hiringDemand.isHighVolumeHiring', 'locale', 'salary', 'expired', 'postedToday', 'scrapingInfo.page', 'scrapingInfo.index', 'jobKey', 'jobUrl', 'emails', 'isRemote', 'socialInsurance', 'shiftAndSchedule', 'applyUrl', 'requirements', 'companyLogoUrl', 'companyHeaderUrl', 'numOfCandidates']

Clean rows: 97
                                               title       company  \
0                      Analyste d'exploitation - F/H       INZERTY   
1  Stage 2 mois – Data Analyst / Lead Generation ...  Vroom Market   
2             Consolideur analyste expérimenté (H/F)   AEMA GROUPE   

          city country      job_type date_published month_posted day_of_week  \
0        Lille  France  Non spécifié     2026-03-05      2026-03    Thursday   
